In [ ]:
import os
import glob
import time
import struct
import socket
import threading
import numpy as np
from pynq import Overlay, allocate, PL

PL.reset()
ol = Overlay("depack_pipeline.bit")

dma_lidar = ol.axi_dma_0
dma_camera = ol.axi_dma_1

print(f"[INFO] dma_lidar.sendchannel max transfer  : {dma_lidar.sendchannel._max_size:,} bytes")
print(f"[INFO] dma_lidar.recvchannel max transfer  : {dma_lidar.recvchannel._max_size:,} bytes")
print(f"[INFO] dma_camera.recvchannel max transfer : {dma_camera.recvchannel._max_size:,} bytes")

MAX_PAYLOAD_BYTES = 1400
NUM_SLOTS = 16
LIDAR_SLOT_SIZE = 1024 * MAX_PAYLOAD_BYTES
TOTAL_LIDAR_BYTES = NUM_SLOTS * LIDAR_SLOT_SIZE
CAMERA_SLOT_SIZE = 1024 * MAX_PAYLOAD_BYTES
TOTAL_CAMERA_BYTES = NUM_SLOTS * CAMERA_SLOT_SIZE

buf_l2_cam = allocate(shape=(TOTAL_CAMERA_BYTES,), dtype=np.uint8, cacheable=False)
buf_l2_lidar = allocate(shape=(TOTAL_LIDAR_BYTES,), dtype=np.uint8, cacheable=False)
buf_l2_cam[:] = 0
buf_l2_lidar[:] = 0

phys_cam = buf_l2_cam.physical_address
phys_lidar = buf_l2_lidar.physical_address

reg_l2 = ol.layer2_depack_0.register_map
ol.layer2_depack_0.write(reg_l2.ddr_mem_camera_1.address, phys_cam & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_camera_2.address, (phys_cam >> 32) & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_lidar_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_lidar_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer2_depack_0.write(0x00, 0x81)

reg_l1_cam = ol.layer1_camera_prep_0.register_map
ol.layer1_camera_prep_0.write(reg_l1_cam.ddr_mem_1.address, phys_cam & 0xFFFFFFFF)
ol.layer1_camera_prep_0.write(reg_l1_cam.ddr_mem_2.address, (phys_cam >> 32) & 0xFFFFFFFF)
ol.layer1_camera_prep_0.write(0x00, 0x81)

reg_l1 = ol.layer1_depack_0.register_map
ol.layer1_depack_0.write(reg_l1.ddr_mem_meta_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_meta_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_points_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_points_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer1_depack_0.write(0x00, 0x81)

ol.packet_dispatcher_0.write(0x00,0x81);


CAMERA_DESC_WORDS = 9
MAX_WORDS_PER_LIDAR_FRAME = 262144
MAX_WORDS_PER_CAMERA_FRAME = (CAMERA_SLOT_SIZE // 4) + CAMERA_DESC_WORDS
INACTIVITY_TIMEOUT_SEC = 3.0

BATCH_PACKET_COUNT = 32
BATCH_FLUSH_TIMEOUT_SEC = 0.005
PACKET_SLOT_BYTES = 1400

PIXEL_FORMAT_NAMES = {0: "RGB", 1: "BGR", 2: "RGBA", 3: "GRAY"}


def dma_channel_error(channel):
    try:
        return bool(channel.error)
    except Exception:
        return None


def wait_for_idle(channel, timeout_s, label):
    t0 = time.time()
    while not channel.idle:
        if dma_channel_error(channel):
            print(f"[ERROR] {label}: DMA reported error.")
            return False
        if time.time() - t0 > timeout_s:
            print(f"[TIMEOUT] {label} timeout.")
            return False
    return True


def unpack_camera_field_desc(words9):
    raw = struct.pack("<9I", *[int(w) for w in words9])
    slot_index = raw[0]
    sensor_id = raw[1]
    frame_id = struct.unpack("<I", raw[2:6])[0]
    valid = raw[6]
    width = struct.unpack("<H", raw[7:9])[0]
    height = struct.unpack("<H", raw[9:11])[0]
    gop_size = raw[11]
    crf = raw[12]
    shared_field = raw[13]

    ts_bytes = raw[14:22]
    try:
        timestamp = struct.unpack(">d", ts_bytes)[0]
    except Exception:
        timestamp = 0.0

    bitstream_addr = struct.unpack("<Q", raw[22:30])[0]
    bitstream_length = struct.unpack("<I", raw[30:34])[0]
    compressed = bool(raw[34] & 0x01)

    profile_id = shared_field if compressed else None
    pixel_format = None if compressed else shared_field

    return {
        "slot_index": slot_index,
        "sensor_id": sensor_id,
        "frame_id": frame_id,
        "valid": valid,
        "compressed": compressed,
        "width": width,
        "height": height,
        "gop": gop_size,
        "crf": crf,
        "profile_id": profile_id,
        "pixel_format": pixel_format,
        "timestamp": timestamp,
        "bitstream_addr": bitstream_addr,
        "bitstream_length": bitstream_length,
    }


def load_golden_packed_words_dict(path):
    golden_dict = {}
    current_frame_id, current_words = None, []
    if not os.path.exists(path):
        return golden_dict
    with open(path, "r") as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            if parts[0] == "FRAME":
                if current_frame_id is not None and current_words:
                    golden_dict[current_frame_id] = np.array(current_words, dtype=np.uint32)
                    current_words = []
                current_frame_id = int(parts[1])
            elif len(parts) == 1:
                current_words.append(int(parts[0], 16))
        if current_frame_id is not None and current_words:
            golden_dict[current_frame_id] = np.array(current_words, dtype=np.uint32)
    return golden_dict


def load_golden_camera_dict(golden_dir):
    cam_dict = {}
    for path in sorted(glob.glob(os.path.join(golden_dir, "golden_camera_*.h264"))):
        fname = os.path.basename(path)
        frame_id = int(fname.replace("golden_camera_", "").replace(".h264", ""))
        with open(path, "rb") as f:
            cam_dict[frame_id] = f.read()
    return cam_dict


def load_golden_camera_raw_dict(golden_dir):
    cam_dict = {}
    for path in sorted(glob.glob(os.path.join(golden_dir, "golden_camera_raw_*.bin"))):
        fname = os.path.basename(path)
        frame_id = int(fname.replace("golden_camera_raw_", "").replace(".bin", ""))
        with open(path, "rb") as f:
            cam_dict[frame_id] = f.read()
    return cam_dict


rx_buf0 = allocate(shape=(MAX_WORDS_PER_LIDAR_FRAME,), dtype=np.uint32, cacheable=False)
rx_buf1 = allocate(shape=(MAX_WORDS_PER_LIDAR_FRAME,), dtype=np.uint32, cacheable=False)
rx_bufs = [rx_buf0, rx_buf1]

cam_rx_buf0 = allocate(shape=(MAX_WORDS_PER_CAMERA_FRAME,), dtype=np.uint32, cacheable=False)
cam_rx_buf1 = allocate(shape=(MAX_WORDS_PER_CAMERA_FRAME,), dtype=np.uint32, cacheable=False)
cam_rx_bufs = [cam_rx_buf0, cam_rx_buf1]

dma_lidar.sendchannel.start()
dma_lidar.recvchannel.start()
dma_camera.recvchannel.start()

rx_active = True
total_bytes_received = 0
first_packet_time = 0.0
last_packet_time = 0.0
send_stalled = False

captured_raw_cam_buffers = []
captured_raw_lidar_buffers = []
active_rx = 0
active_cam_rx = 0

lidar_transfer_pending = True
cam_transfer_pending = True

dma_lidar.recvchannel.transfer(rx_bufs[active_rx])
dma_camera.recvchannel.transfer(cam_rx_bufs[active_cam_rx])


def udp_rx_dma_worker():
    global rx_active, total_bytes_received, first_packet_time, last_packet_time, send_stalled

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEPORT, 1)
    except (AttributeError, OSError):
        pass
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, 32 * 1024 * 1024)

    try:
        sock.bind(("0.0.0.0", 6000))
    except OSError as e:
        print(f"[NET WORKER ERROR] bind() failed: {e}")
        sock.close()
        rx_active = False
        return

    print("[NET WORKER] Listening on UDP port 6000...")

    tx_dma_buf = allocate(shape=(BATCH_PACKET_COUNT * PACKET_SLOT_BYTES,),
                          dtype=np.uint8, cacheable=False)

    batch_chunks = []
    batch_count = 0
    batch_deadline = None
    dma_in_flight = False
    last_packet_time = time.time()

    def add_to_batch(pkt_bytes):
        nonlocal batch_chunks, batch_count, batch_deadline
        length = len(pkt_bytes)
        pad = (-length) % 4
        batch_chunks.append(pkt_bytes)
        if pad:
            batch_chunks.append(b'\x00' * pad)
        batch_count += 1
        if batch_deadline is None:
            batch_deadline = time.time() + BATCH_FLUSH_TIMEOUT_SEC

    def flush_batch():
        global send_stalled
        nonlocal batch_chunks, batch_count, batch_deadline, dma_in_flight
        if batch_count == 0:
            return
        if dma_in_flight and not wait_for_idle(dma_lidar.sendchannel, 2.0, "sendchannel"):
            send_stalled = True
            batch_chunks, batch_count, batch_deadline = [], 0, None
            return

        blob = b"".join(batch_chunks)
        tx_dma_buf[:len(blob)] = np.frombuffer(blob, dtype=np.uint8)
        dma_lidar.sendchannel.transfer(tx_dma_buf[:len(blob)])
        dma_in_flight = True

        batch_chunks, batch_count, batch_deadline = [], 0, None

    try:
        while rx_active:
            try:
                timeout = 0.5
                if batch_deadline is not None:
                    timeout = max(0.0001, min(timeout, batch_deadline - time.time()))
                sock.settimeout(timeout)

                data, addr = sock.recvfrom(2048)
                current_time = time.time()
                last_packet_time = current_time
                if first_packet_time == 0.0:
                    first_packet_time = current_time

                if data == b"EOF_STREAM":
                    flush_batch()
                    if dma_in_flight:
                        if not wait_for_idle(dma_lidar.sendchannel, 2.0, "sendchannel (EOF flush)"):
                            send_stalled = True
                    print("[NET WORKER] EOF Stream signal received.")
                    break

                total_bytes_received += len(data)
                add_to_batch(data)

                if batch_count >= BATCH_PACKET_COUNT:
                    flush_batch()

            except socket.timeout:
                if batch_deadline is not None and time.time() >= batch_deadline:
                    flush_batch()
                if first_packet_time > 0.0 and (time.time() - last_packet_time >= INACTIVITY_TIMEOUT_SEC):
                    print(f"[NET WORKER] No packets received for {INACTIVITY_TIMEOUT_SEC}s. Stopping stream.")
                    flush_batch()
                    break
                continue
            except Exception as e:
                print(f"[NET WORKER ERROR] {e}")
                break
    finally:
        try:
            flush_batch()
        except Exception as e:
            print(f"[NET WORKER] Final flush error: {e}")
        try:
            sock.close()
        except Exception:
            pass
        print("[NET WORKER] Network thread finished.")
        
def process_camera_completion():
    global active_cam_rx, cam_transfer_pending

    if not cam_transfer_pending:
        return
    if not dma_camera.recvchannel.idle:
        return

    curr_buf = cam_rx_bufs[active_cam_rx]
    if hasattr(curr_buf, "invalidate"):
        curr_buf.invalidate()

    captured_raw_cam_buffers.append(curr_buf.copy())
    cam_transfer_pending = False

    active_cam_rx = 1 - active_cam_rx
    if rx_active:
        dma_camera.recvchannel.transfer(cam_rx_bufs[active_cam_rx])
        cam_transfer_pending = True


def process_lidar_completion():
    global active_rx, lidar_transfer_pending

    if not lidar_transfer_pending:
        return
    if not dma_lidar.recvchannel.idle:
        return

    curr_buf = rx_bufs[active_rx]
    if hasattr(curr_buf, "invalidate"):
        curr_buf.invalidate()

    captured_raw_lidar_buffers.append(curr_buf.copy())
    lidar_transfer_pending = False

    active_rx = 1 - active_rx
    if rx_active:
        dma_lidar.recvchannel.transfer(rx_bufs[active_rx])
        lidar_transfer_pending = True


def service_hardware():
    process_camera_completion()
    process_lidar_completion()


net_thread = threading.Thread(target=udp_rx_dma_worker, daemon=True)
net_thread.start()

while net_thread.is_alive():
    service_hardware()
    time.sleep(0.0001)

rx_active = False

t_drain = time.time()
while time.time() - t_drain < 1.0:
    service_hardware()
    time.sleep(0.0005)

net_thread.join()

total_exec_time = (last_packet_time - first_packet_time) if (first_packet_time and last_packet_time) else 0.0

golden_dir = "golden_outputs"
golden_lidar_dict = load_golden_packed_words_dict(os.path.join(golden_dir, "golden_lidar.dat"))
golden_camera_dict = load_golden_camera_dict(golden_dir)
golden_camera_raw_dict = load_golden_camera_raw_dict(golden_dir)

print("\n" + "=" * 70)
print(" RUNNING DEFERRED DECODING & GOLDEN VERIFICATION")
print("=" * 70)

total_cam_passed = 0
total_cam_compressed_passed = 0
total_cam_raw_passed = 0

print("\n--- DECODING & VERIFYING CAMERA FRAMES ---")
for idx, raw_buf in enumerate(captured_raw_cam_buffers):
    fd = unpack_camera_field_desc(raw_buf[0:CAMERA_DESC_WORDS])
    frame_id = fd["frame_id"]
    is_compressed = fd["compressed"]
    bitstream_length = fd["bitstream_length"]
    bitstream_word_count = (bitstream_length + 3) // 4

    if CAMERA_DESC_WORDS + bitstream_word_count > len(raw_buf):
        print(f"   [CAM OUT] Frame {idx:2d} (ID {frame_id}): [ERROR] Corrupted descriptor (Length={bitstream_length} B)")
        continue

    bitstream_words = raw_buf[CAMERA_DESC_WORDS : CAMERA_DESC_WORDS + bitstream_word_count]
    extracted_payload = bitstream_words.tobytes()[:bitstream_length]

    golden_bytes = (golden_camera_dict.get(frame_id) if is_compressed else golden_camera_raw_dict.get(frame_id))

    if golden_bytes is None:
        kind = "H.264" if is_compressed else "raw"
        print(f"   [CAM OUT] Frame ID {frame_id:2d}: [WARN] Reference golden {kind} file missing")
    elif extracted_payload == golden_bytes:
        total_cam_passed += 1
        if is_compressed:
            total_cam_compressed_passed += 1
        else:
            total_cam_raw_passed += 1
        print(f"   [CAM OUT] Frame ID {frame_id:2d} ({'COMPRESSED' if is_compressed else 'RAW'}): "
              f"[PASSED] 100% Match ({len(extracted_payload):,} B)")
    else:
        print(f"   [CAM OUT] Frame ID {frame_id:2d} ({'COMPRESSED' if is_compressed else 'RAW'}): "
              f"[MISMATCH] Rx Len: {len(extracted_payload):,} B vs Golden Len: {len(golden_bytes):,} B")

total_lidar_passed = 0
print("\n--- DECODING & VERIFYING LIDAR FRAMES ---")
for idx, raw_buf in enumerate(captured_raw_lidar_buffers):
    raw_w0 = int(raw_buf[0])
    if raw_w0 == 0:
        continue

    b0 = raw_w0 & 0xFF
    b1 = (raw_w0 >> 8) & 0xFF
    b2 = (raw_w0 >> 16) & 0xFF
    b3 = (raw_w0 >> 24) & 0xFF
    hw_frame_id = (b0 << 24) | (b1 << 16) | (b2 << 8) | b3

    if hw_frame_id not in golden_lidar_dict:
        alt_id = int(raw_w0 & 0xFFFF)
        if alt_id in golden_lidar_dict:
            hw_frame_id = alt_id

    golden_payload = golden_lidar_dict.get(hw_frame_id)

    if golden_payload is not None:
        expected_len = len(golden_payload)
        hw_payload = raw_buf[0:expected_len]

        if len(hw_payload) == len(golden_payload) and np.array_equal(hw_payload, golden_payload):
            total_lidar_passed += 1
            print(f"   [LIDAR OUT] Frame ID {hw_frame_id:2d} ({expected_len:,} words): [PASSED] 100% Word Match")
        else:
            mismatches = np.where(hw_payload != golden_payload[:len(hw_payload)])[0]
            print(f"   [LIDAR OUT] Frame ID {hw_frame_id:2d}: [MISMATCH] ({len(mismatches)}/{expected_len} words failed)")
    else:
        print(f"   [LIDAR OUT] Frame ID {hw_frame_id:2d} (Raw W0: 0x{raw_w0:08X}): [WARN] Golden reference missing")

print("\n" + "=" * 70)
print(" FINAL BENCHMARK SUMMARY")
print("=" * 70)
print(f" Send Stalled          : {send_stalled}")
print(f" Total Stream Time     : {total_exec_time:.3f} s")
print(f" Total Bytes Received  : {total_bytes_received:,} B")
print(f" Camera Frames Captured: {len(captured_raw_cam_buffers)}")
print(f" Camera Frame Results  : {total_cam_passed}/{len(captured_raw_cam_buffers)} Passed "
      f"(Compressed: {total_cam_compressed_passed}, Raw: {total_cam_raw_passed})")
print(f" LiDAR Frames Captured : {len(captured_raw_lidar_buffers)}")
print(f" LiDAR Frame Results   : {total_lidar_passed}/{len(captured_raw_lidar_buffers)} Passed")
print("=" * 70)

if (len(captured_raw_cam_buffers) != len(captured_raw_lidar_buffers)
        or len(captured_raw_cam_buffers) == 0):
    print(f"[WARN] Camera ({len(captured_raw_cam_buffers)}) and LiDAR "
          f"({len(captured_raw_lidar_buffers)}) capture counts don't match "
          f"the expected 1:1 frame pairing - check for duplicate/dropped captures.")

[INFO] dma_lidar.sendchannel max transfer  : 67,108,863 bytes
[INFO] dma_lidar.recvchannel max transfer  : 67,108,863 bytes
[INFO] dma_camera.recvchannel max transfer : 67,108,863 bytes
[NET WORKER] Listening on UDP port 6000...
[NET WORKER] EOF Stream signal received.
[NET WORKER] Network thread finished.

 RUNNING DEFERRED DECODING & GOLDEN VERIFICATION

--- DECODING & VERIFYING CAMERA FRAMES ---
   [CAM OUT] Frame ID 2000 (COMPRESSED): [PASSED] 100% Match (244,016 B)
   [CAM OUT] Frame ID 2001 (COMPRESSED): [PASSED] 100% Match (201,892 B)
   [CAM OUT] Frame ID 2002 (COMPRESSED): [PASSED] 100% Match (244,016 B)
   [CAM OUT] Frame ID 2003 (COMPRESSED): [PASSED] 100% Match (201,892 B)
   [CAM OUT] Frame ID 2004 (COMPRESSED): [PASSED] 100% Match (244,016 B)
   [CAM OUT] Frame ID 2005 (COMPRESSED): [PASSED] 100% Match (201,892 B)
   [CAM OUT] Frame ID 2006 (COMPRESSED): [PASSED] 100% Match (244,016 B)
   [CAM OUT] Frame ID 2007 (COMPRESSED): [PASSED] 100% Match (201,892 B)
   [CAM OUT] F